# Acidentes no Espaço Aéreo Brasileiro (Cenipa)
Projeto AC2 - Big Data

## Configuração do Dataset

In [47]:
# Importando bibliotecas necessárias
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, isnan, when
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.classification import DecisionTreeClassifier, LogisticRegression, MultilayerPerceptronClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import unicodedata
import re

spark = SparkSession.builder \
    .appName("Cenipa_BigData_Classification") \
    .getOrCreate()

In [48]:
dataframe = spark.read.csv("./assets/data/Cenipa.csv", header=True, inferSchema=True, sep=";")

num_linhas = dataframe.count()
print(f"Número de linhas no DataFrame: {num_linhas}")

Número de linhas no DataFrame: 6114


## Análise de Dados

In [40]:
# Visualizando o esquema dos dados
dataframe.printSchema()

root
 |-- Codigo da Ocorrencia: integer (nullable = true)
 |-- Classificacao da Ocorrencia : string (nullable = true)
 |-- Data e Hora da Ocorrencia: string (nullable = true)
 |-- Latitude da Ocorrencia: string (nullable = true)
 |-- Longitude da Ocorrencia: string (nullable = true)
 |-- Cidade da Ocorrencia: string (nullable = true)
 |-- UF da Ocorrencia: string (nullable = true)
 |-- Pais da Ocorrencia: string (nullable = true)
 |-- Aerodromo da Ocorrencia: string (nullable = true)
 |-- Investigacao da Aeronave foi Liberada: string (nullable = true)
 |-- Status da Investigacao: string (nullable = true)
 |-- Numero do Relatorio de Divulgacao: string (nullable = true)
 |-- Relatorio de  Divulgacao foi Publicado?: string (nullable = true)
 |-- Dia da Divulgacao da Publicacao: string (nullable = true)
 |-- Total de Recomendacoes: integer (nullable = true)
 |-- Total de Aeronaves Envolvidas: integer (nullable = true)
 |-- Ocorrencia na Saida da Pista?: string (nullable = true)
 |-- Tipo d

In [50]:
# Função para limpar e normalizar strings
def normalizar_nome_coluna(nome):
    # Remove acentos
    nome_sem_acento = unicodedata.normalize('NFKD', nome).encode('ASCII', 'ignore').decode('utf-8')
    # Substitui espaços e hifens por underline (_) e deixa tudo em minúsculo
    nome_formatado = re.sub(r'[\s\-]+', '_', nome_sem_acento.strip()).lower()
    # Remove quaisquer outros caracteres especiais que tenham sobrado
    nome_limpo = re.sub(r'[^\w]', '', nome_formatado)
    return nome_limpo

# Cria uma lista com os novos nomes das colunas
novas_colunas = [normalizar_nome_coluna(c) for c in dataframe.columns]

# Aplica os novos nomes ao DataFrame
dataframe = dataframe.toDF(*novas_colunas)

print("Novas colunas normalizadas:")
print(dataframe.columns)

Novas colunas normalizadas:
['codigo_da_ocorrencia', 'classificacao_da_ocorrencia', 'data_e_hora_da_ocorrencia', 'latitude_da_ocorrencia', 'longitude_da_ocorrencia', 'cidade_da_ocorrencia', 'uf_da_ocorrencia', 'pais_da_ocorrencia', 'aerodromo_da_ocorrencia', 'investigacao_da_aeronave_foi_liberada', 'status_da_investigacao', 'numero_do_relatorio_de_divulgacao', 'relatorio_de_divulgacao_foi_publicado', 'dia_da_divulgacao_da_publicacao', 'total_de_recomendacoes', 'total_de_aeronaves_envolvidas', 'ocorrencia_na_saida_da_pista', 'tipo_de_ocorrencia', 'tipo_de_categoria_da_ocorrencia', 'taxonomia_do_tipo_de_icao', 'matricula_da_aeronave', 'categoria_do_operador_de_aeronave', 'tipo_de_aeronave', 'fabricante_da_aeronave', 'modelo_de_aeronave', 'tipo_de_aeronave_icao', 'aeronave_motor_tipo', 'aeronave_motor_quantidade', 'aeronave_pmd', 'categoria_pmd_aeronave', 'quantidade_de_assentos_na_aeronave', 'ano_de_fabricacao_da_aeronave', 'pais_fabricante_da_aeronave', 'pais_de_registro_da_aeronave', '

In [51]:
dataframe.printSchema()

root
 |-- codigo_da_ocorrencia: integer (nullable = true)
 |-- classificacao_da_ocorrencia: string (nullable = true)
 |-- data_e_hora_da_ocorrencia: string (nullable = true)
 |-- latitude_da_ocorrencia: string (nullable = true)
 |-- longitude_da_ocorrencia: string (nullable = true)
 |-- cidade_da_ocorrencia: string (nullable = true)
 |-- uf_da_ocorrencia: string (nullable = true)
 |-- pais_da_ocorrencia: string (nullable = true)
 |-- aerodromo_da_ocorrencia: string (nullable = true)
 |-- investigacao_da_aeronave_foi_liberada: string (nullable = true)
 |-- status_da_investigacao: string (nullable = true)
 |-- numero_do_relatorio_de_divulgacao: string (nullable = true)
 |-- relatorio_de_divulgacao_foi_publicado: string (nullable = true)
 |-- dia_da_divulgacao_da_publicacao: string (nullable = true)
 |-- total_de_recomendacoes: integer (nullable = true)
 |-- total_de_aeronaves_envolvidas: integer (nullable = true)
 |-- ocorrencia_na_saida_da_pista: string (nullable = true)
 |-- tipo_de_oc

In [52]:
dataframe.show(5)

+--------------------+---------------------------+-------------------------+----------------------+-----------------------+--------------------+----------------+------------------+-----------------------+-------------------------------------+----------------------+---------------------------------+-------------------------------------+-------------------------------+----------------------+-----------------------------+----------------------------+--------------------+-------------------------------+-------------------------+---------------------+---------------------------------+----------------+----------------------+------------------+---------------------+-------------------+-------------------------+------------+----------------------+----------------------------------+-----------------------------+---------------------------+----------------------------+---------------------------------+-----------------------------+-------------------------+-----------------------+---------------

In [54]:
# Verificando a distribuição da classe alvo "Classificacao da Ocorrencia"
print("Distribuição da classe alvo (Classificacao da Ocorrencia):")
distribuicao_classes = dataframe.groupBy("classificacao_da_ocorrencia").count().orderBy(col("count").desc())
distribuicao_classes.show()

Distribuição da classe alvo (Classificacao da Ocorrencia):
+---------------------------+-----+
|classificacao_da_ocorrencia|count|
+---------------------------+-----+
|                  INCIDENTE| 3393|
|                   ACIDENTE| 1930|
|            INCIDENTE GRAVE|  791|
+---------------------------+-----+



## Limpando o Dataset

In [43]:
# Tratando valores nulos
# dataframe = dataframe.dropna()

num_linhas = dataframe.count()
print(f"Número de linhas no DataFrame: {num_linhas}")

Número de linhas no DataFrame: 6114


In [44]:
dataframe.printSchema()

root
 |-- Codigo da Ocorrencia: integer (nullable = true)
 |-- Classificacao da Ocorrencia : string (nullable = true)
 |-- Data e Hora da Ocorrencia: string (nullable = true)
 |-- Latitude da Ocorrencia: string (nullable = true)
 |-- Longitude da Ocorrencia: string (nullable = true)
 |-- Cidade da Ocorrencia: string (nullable = true)
 |-- UF da Ocorrencia: string (nullable = true)
 |-- Pais da Ocorrencia: string (nullable = true)
 |-- Aerodromo da Ocorrencia: string (nullable = true)
 |-- Investigacao da Aeronave foi Liberada: string (nullable = true)
 |-- Status da Investigacao: string (nullable = true)
 |-- Numero do Relatorio de Divulgacao: string (nullable = true)
 |-- Relatorio de  Divulgacao foi Publicado?: string (nullable = true)
 |-- Dia da Divulgacao da Publicacao: string (nullable = true)
 |-- Total de Recomendacoes: integer (nullable = true)
 |-- Total de Aeronaves Envolvidas: integer (nullable = true)
 |-- Ocorrencia na Saida da Pista?: string (nullable = true)
 |-- Tipo d

## Pré-Processamento

In [56]:
# 1. Removendo registros nulos na coluna alvo
df_clean = dataframe.dropna(subset=["classificacao_da_ocorrencia"])

# 2. Convertendo a coluna alvo (String) para valores numéricos (Double)
label_indexer = StringIndexer(inputCol="classificacao_da_ocorrencia", outputCol="label")
df_clean = label_indexer.fit(df_clean).transform(df_clean)

# 3. Tratamento de classes desbalanceadas (Undersampling)
# Calculamos a proporção para igualar todas as classes pela classe minoritária
class_counts = df_clean.groupBy('label').count().collect()
min_count = min([row['count'] for row in class_counts])
fractions = {row['label']: min_count / row['count'] for row in class_counts}

df_balanced = df_clean.sampleBy('label', fractions, seed=42)
print("Distribuição após o balanceamento das classes:")
df_balanced.groupBy("label").count().show()

Distribuição após o balanceamento das classes:
+-----+-----+
|label|count|
+-----+-----+
|  0.0|  796|
|  1.0|  821|
|  2.0|  791|
+-----+-----+

